# Week 20 · Notebook 01: Bedrock Converse & Knowledge Bases

# Requirements: pip install boto3 numpy pandas

# ⚠️ REQUIRES: AWS credentials + Bedrock model access

> 💰 COST WARNING: set an AWS budget alert before running, Bedrock bills per token and Knowledge Bases bill per query + storage.


## What you build

Call Bedrock models with the provider-neutral **Converse API** (run the Week 6 prompt suite across two models and compare), then query a **Knowledge Base** for RAG on the shipping-policy docs with citations, ending by printing recall and groundedness. Dry-run mode uses a local retriever so the metrics are reproducible without AWS.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1])) # repo root (template)
# Robust fallback: walk up until we find zoro/data.py, in case Jupyter started elsewhere.
_root = pathlib.Path.cwd()
for _p in [_root, *_root.parents]:
    if (_p / "zoro" / "data.py").exists():
        sys.path.insert(0, str(_p))
        break

import os
import json
import numpy as np
import pandas as pd
import boto3
from zoro import data

SEED = 20
rng = np.random.default_rng(SEED)
REGION = os.environ.get("AWS_REGION", "us-east-1")

def probe_aws():
    try:
        boto3.client("bedrock", region_name=REGION).list_foundation_models()
        return True
    except Exception as e: # noqa: BLE001
        print("AWS probe failed:", type(e).__name__, e)
        return False

BEDROCK_READY = probe_aws()
if not BEDROCK_READY:
    print("⚠️ AWS credentials not found, running in DRY-RUN mode.")
    print("Run `aws configure` (or `aws sso login`), then restart the kernel.")
    print("Also enable models under Bedrock → Model access (AccessDeniedException = forgot this).")

runtime = boto3.client("bedrock-runtime", region_name=REGION)
print("boto3 ready; region =", REGION)


## Converse API: one surface, many models

The Converse API is the cleanest provider-neutral abstraction of the three clouds: swap the `modelId` and nothing else. Model IDs carry dated suffixes and get superseded often, copy the current ID from the console.


In [ ]:
MODELS = [
    os.environ.get("BEDROCK_MODEL_A", "amazon.nova-lite-v1:0"),
    os.environ.get("BEDROCK_MODEL_B", "anthropic.claude-3-5-sonnet-20241022-v2:0"),
]

def converse(prompt, model_id):
    resp = runtime.converse(
        modelId=model_id,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
    )
    return resp["output"]["message"]["content"][0]["text"]

if BEDROCK_READY:
    try:
        print("Smoke test:", converse("Reply with exactly: OK", MODELS[0])[:60])
    except Exception as e:  # noqa: BLE001
        print("Converse smoke test failed (model not enabled?):", type(e).__name__, e)
else:
    print("DRY-RUN: skipped Converse smoke test (no AWS credentials).")


## Run the Week 6 prompt suite across 2 models

Same BoL extraction prompt, two model IDs, same seed. Per-field accuracy lets you pick a model with data, not vibes. Keep the cloud run to 2 docs per model until you confirm cost.


In [ ]:
bols = data.bol_samples(n=4, seed=5)

EXTRACT_PROMPT = (
    "Extract the bill-of-lading fields from the text below.\n"
    "Return ONLY valid JSON with exactly these keys:\n"
    "shipper, consignee, port_of_loading, port_of_discharge, commodity,\n"
    "quantity, gross_weight_kg, declared_value_usd, freight_terms, date_of_issue.\n"
    "quantity and gross_weight_kg must be integers; declared_value_usd a number.\n\n"
    "TEXT:\n{text}"
)

def normalize_string(s):
    return str(s).strip().lower()

def field_matches(pred, truth, key):
    if pred is None or truth is None:
        return False
    if key in ("quantity", "gross_weight_kg"):
        try:
            return int(float(pred)) == int(float(truth))
        except (TypeError, ValueError):
            return False
    if key == "declared_value_usd":
        try:
            return abs(float(pred) - float(truth)) < 0.01
        except (TypeError, ValueError):
            return False
    return normalize_string(pred) == normalize_string(truth)

FIELDS = ["shipper", "consignee", "port_of_loading", "port_of_discharge",
          "commodity", "quantity", "gross_weight_kg", "declared_value_usd",
          "freight_terms", "date_of_issue"]

def accuracy_for(model_id, docs):
    correct = total = 0
    for b in docs:
        raw = ""
        try:
            raw = converse(EXTRACT_PROMPT.format(text=b["text"]), model_id).strip()
            raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
            pred = json.loads(raw)
        except Exception as e:  # noqa: BLE001
            pred = None
            print("  failed on", b["bol_id"], "->", type(e).__name__)
        for f in FIELDS:
            total += 1
            if pred and field_matches(pred.get(f), b["fields"].get(f), f):
                correct += 1
    return correct / total if total else 0.0

results = []
for m in MODELS:
    if BEDROCK_READY:
        acc = accuracy_for(m, bols[:2])
    else:
        acc = 0.0
        print(f"DRY-RUN: no AWS call for {m} (accuracy set to 0.0).")
    results.append({"model": m, "accuracy": acc})

model_df = pd.DataFrame(results)
print(model_df.to_string(index=False))


## Knowledge Bases (RAG)

Point Bedrock at S3, and it chunks, embeds, and exposes `retrieve_and_generate` with citations. Set `BEDROCK_KB_ID` (and `BEDROCK_MODEL_ARN` for the generator model) after creating the KB in the console. Dry-run uses a local retriever over `data.policy_docs()`.


In [ ]:
kb_id = os.environ.get("BEDROCK_KB_ID", "")
model_arn = os.environ.get("BEDROCK_MODEL_ARN", "")
kb_client = boto3.client("bedrock-agent-runtime", region_name=REGION)

def kb_retrieve_and_generate(question):
    resp = kb_client.retrieve_and_generate(
        input={"text": question},
        retrieveAndGenerateConfiguration={
            "type": "KNOWLEDGE_BASE",
            "knowledgeBaseConfiguration": {
                "knowledgeBaseId": kb_id,
                "modelArn": model_arn,
            },
        },
    )
    return resp["output"]["text"], resp.get("citations", [])

policies = data.policy_docs()
RAG_KEYWORDS = {"refund": "POL-002", "damaged": "POL-002", "address": "POL-001",
                "dangerous": "POL-003", "customs": "POL-004", "storage": "POL-004"}

def local_retrieve(question):
    q = question.lower()
    doc_id = next((d for k, d in RAG_KEYWORDS.items() if k in q), None)
    if doc_id:
        for d in policies:
            if d["doc_id"] == doc_id:
                return d["text"], [{"doc_id": doc_id}]
    return "", []


## Measure recall & groundedness

**Recall** = did we retrieve the right source document? **Groundedness** = does the answer actually contain the ground-truth fact (i.e. come from a real citation, not a hallucination)? Both are fractions over the question set.


In [ ]:
rag_questions = [
    {"id": "R1", "question": "What is the refund policy for damaged freight?", "doc_id": "POL-002", "expect": ["$5,000"]},
    {"id": "R2", "question": "What is the address-change fee after pickup?", "doc_id": "POL-001", "expect": ["$85"]},
    {"id": "R3", "question": "What documents does a dangerous-goods shipment need?", "doc_id": "POL-003", "expect": ["UN number"]},
    {"id": "R4", "question": "What is the customs hold storage fee beyond 5 days?", "doc_id": "POL-004", "expect": ["$40"]},
]

rows, recall_hits, grounded_hits = [], 0, 0
for item in rag_questions:
    if BEDROCK_READY and kb_id and model_arn:
        text, citations = kb_retrieve_and_generate(item["question"])
        cited = bool(citations)
        grounded = any(k.lower() in text.lower() for k in item["expect"])
    else:
        text, citations = local_retrieve(item["question"])
        cited = any(c.get("doc_id") == item["doc_id"] for c in citations)
        grounded = any(k.lower() in text.lower() for k in item["expect"])
    recall_hits += int(cited)
    grounded_hits += int(grounded)
    rows.append({"id": item["id"], "cited": cited, "grounded": grounded})

rag_df = pd.DataFrame(rows)
n = len(rag_questions)
recall = recall_hits / n if n else 0.0
groundedness = grounded_hits / n if n else 0.0
print(rag_df.to_string(index=False))


In [ ]:
# Final numbers: RAG recall and groundedness over the question set.
print(f"RECALL: {recall:.3f}  GROUNDEDNESS: {groundedness:.3f}")
